## 쉐포라 데이터로 ml_v4 재튜닝
### 변경 사항
* 피처 : 가격(고/중/저), 용량, 성분(k뷰티 대표 성분, us 트렌드 성분, spf_value), 카테고리 (스킨, 선크림, 마스크 , 클렌징 ), 중분류 카테고리
* 성분 TF-IDF 제거 
* 성분 position 관련 피처 튜닝 예정

In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, QuantileTransformer
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict, RandomizedSearchCV, GridSearchCV
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from lightgbm import LGBMClassifier
import re, warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

DATA_PATH = r"C:\workspace\finalproject\data\sephora_preprocessed_final.csv"
df = pd.read_csv(DATA_PATH, encoding='cp949', sep='\t', low_memory=False)

print(f"Sephora: {len(df)}개")
print(f"컬럼 수: {len(df.columns)}개")
print(f"sentiment_score 있음: {df['sentiment_score'].notna().sum()}개")
print(f"리뷰수 있음: {df['리뷰수'].notna().sum()}개")
print(f"\ntarget_category 분포:")
print(df['target_category'].value_counts())
print(f"\n카테고리(중) 분포:")
print(df['카테고리(중)'].value_counts())

Sephora: 1802개
컬럼 수: 133개
sentiment_score 있음: 1767개
리뷰수 있음: 1767개

target_category 분포:
target_category
skincare     1182
cleansing     354
masks         161
suncare       105
Name: count, dtype: int64

카테고리(중) 분포:
카테고리(중)
cream                    491
essence/serum/ampoule    459
cleansing foam/gel       197
eye care                 167
wash-off pack            153
sun cream                101
skin/toner                67
mist/fixer                29
cleansing milk/cream      26
cleansing oil             25
lotion                    19
patch                     18
cleansing balm            18
lip&eye remover           17
sheet mask                 7
cleansing tissue/pad       4
sun stick                  4
Name: count, dtype: int64


In [15]:
# ── Y값: PCA(y1 감성 + y2 로그리뷰수) → QuantileTransformer → 극단 30% 이진분류
# sentiment_score, 리뷰수 이미 상품별 집계 완료

df_y = df[df['sentiment_score'].notna() & df['리뷰수'].notna()].copy()

df_y['y1'] = df_y['sentiment_score']
df_y['y2'] = np.log1p(df_y['리뷰수']) / np.log1p(df_y['리뷰수'].max())

scaler_y = StandardScaler()
y_scaled = scaler_y.fit_transform(df_y[['y1', 'y2']])

pca = PCA(n_components=1)
y_pca = pca.fit_transform(y_scaled)
print(f"PCA 분산 설명력: {pca.explained_variance_ratio_[0]:.3f}")
print(f"PC1 로딩: y1(감성)={pca.components_[0,0]:.3f}  y2(리뷰수)={pca.components_[0,1]:.3f}")

qt = QuantileTransformer(output_distribution='uniform', random_state=42)
df_y['y_pca'] = qt.fit_transform(y_pca)

th_lo = df_y['y_pca'].quantile(0.30)
th_hi = df_y['y_pca'].quantile(0.70)
mask  = (df_y['y_pca'] <= th_lo) | (df_y['y_pca'] >= th_hi)
df_y  = df_y[mask].copy()
df_y['target'] = (df_y['y_pca'] >= th_hi).astype(int)

print(f"\n학습 데이터: {len(df_y)}개  (긍정={df_y['target'].sum()}, 부정={len(df_y)-df_y['target'].sum()})")

PCA 분산 설명력: 0.579
PC1 로딩: y1(감성)=0.707  y2(리뷰수)=0.707

학습 데이터: 1060개  (긍정=530, 부정=530)


In [16]:
# ── 피처 생성
ing_cols = [c for c in df_y.columns if c.startswith('성분_')]
df_y['ingredient_text'] = df_y[ing_cols].fillna('').agg(' '.join, axis=1).str.strip()
ing_lower = df_y['ingredient_text'].str.lower()

# 용량(ml)
def extract_ml(s):
    if pd.isna(s): return np.nan
    m = re.search(r'([\d.]+)\s*ml', str(s), re.IGNORECASE)
    if m: return float(m.group(1))
    m = re.search(r'([\d.]+)\s*oz', str(s), re.IGNORECASE)
    if m: return float(m.group(1)) * 29.5735
    return np.nan
df_y['volume_ml'] = df_y['용량'].apply(extract_ml)

# 가격대
price_d = pd.get_dummies(
    pd.cut(df_y['공급가(USD)'], bins=[0,30,71,9999], labels=['low','mid','high']),
    prefix='price'
).astype(int)
df_y = pd.concat([df_y, price_d], axis=1)

# 카테고리(대) 원핫
cat_d = pd.get_dummies(df_y['target_category'], prefix='cat').astype(int)
df_y = pd.concat([df_y, cat_d], axis=1)

# 카테고리(중) 원핫 (희소 제거: n<30)
mid_counts = df_y['카테고리(중)'].value_counts()
valid_mid  = mid_counts[mid_counts >= 30].index
df_y['cat_mid'] = df_y['카테고리(중)'].where(df_y['카테고리(중)'].isin(valid_mid), other='기타')
mid_d = pd.get_dummies(df_y['cat_mid'], prefix='mid').astype(int)
df_y = pd.concat([df_y, mid_d], axis=1)

# K뷰티 성분
kbeauty_map = {
    'k_galactomyces': 'galactomyces', 'k_bifida':      'bifida',
    'k_centella':     'centella',     'k_ginseng':     'ginseng',
    'k_snail':        'snail',        'k_beta_glucan': 'beta-glucan',
    'k_bakuchiol':    'bakuchiol',    'k_pdrn':        'polydeoxyribonucleotide|pdrn',
    'k_propolis':     'propolis',
}
for feat, kw in kbeauty_map.items():
    df_y[feat] = ing_lower.str.contains(kw, regex=True).astype(int)
k_cols = list(kbeauty_map.keys())
df_y['k_beauty_ratio'] = df_y[k_cols].sum(axis=1) / len(k_cols)

# US 트렌드 성분 20개 (EDA 기반 선정)
gt_map = {
    'gt_tocopherol':         'tocopherol',
    'gt_tocopheryl_acetate': 'tocopheryl acetate',
    'gt_sodium_hyaluronate': 'sodium hyaluronate',
    'gt_niacinamide':        'niacinamide',
    'gt_ceramide':           'ceramide',
    'gt_panthenol':          'panthenol',
    'gt_caprylic':           'caprylic',
    'gt_pdrn':               'polydeoxyribonucleotide',
    'gt_exosome':            'exosome',
    'gt_nad':                'nicotinamide adenine',
    'gt_bakuchiol':          'bakuchiol',
    'gt_azelaic_acid':       'azelaic acid',
    'gt_tranexamic_acid':    'tranexamic acid',
    'gt_vitamin_c':          'ascorbic',
    'gt_ectoin':             'ectoin',
    'gt_centella_asiatica':  'centella asiatica',
    'gt_peptide':            'peptide',
    'gt_squalane':           'squalane',
    'gt_caffeine':           'caffeine',
    'gt_retinol':            'retinol',
}
for feat, kw in gt_map.items():
    df_y[feat] = ing_lower.str.contains(kw, regex=True).astype(int)
gt_cols = list(gt_map.keys())
df_y['us_trend_ratio'] = df_y[gt_cols].sum(axis=1) / len(gt_cols)

print("피처 생성 완료")
print(f"카테고리(중) 유효 범주 (n≥30): {valid_mid.tolist()}")
print(f"GT 성분 커버리지:")
for c in gt_cols:
    print(f"  {c}: {df_y[c].mean()*100:.1f}%")

피처 생성 완료
카테고리(중) 유효 범주 (n≥30): ['cream', 'essence/serum/ampoule', 'cleansing foam/gel', 'eye care', 'wash-off pack', 'sun cream', 'skin/toner']
GT 성분 커버리지:
  gt_tocopherol: 44.3%
  gt_tocopheryl_acetate: 27.9%
  gt_sodium_hyaluronate: 43.0%
  gt_niacinamide: 15.8%
  gt_ceramide: 10.0%
  gt_panthenol: 15.0%
  gt_caprylic: 33.6%
  gt_pdrn: 0.0%
  gt_exosome: 0.0%
  gt_nad: 0.0%
  gt_bakuchiol: 2.7%
  gt_azelaic_acid: 1.0%
  gt_tranexamic_acid: 0.8%
  gt_vitamin_c: 10.7%
  gt_ectoin: 0.4%
  gt_centella_asiatica: 4.0%
  gt_peptide: 20.1%
  gt_squalane: 24.9%
  gt_caffeine: 13.8%
  gt_retinol: 4.7%


In [17]:
# ── 성분 Position 파생 피처 (GT 검색량 상위 5개)
# 농도 내림차순 의무 표기 활용: 성분 위치 = 농도 프록시

position_targets = {
    'niacinamide': 'niacinamide',
    'retinol':     'retinol',
    'vitamin_c':   'ascorbic',
    'ceramide':    'ceramide',
    'peptide':     'peptide',
}

total_ing = df_y[ing_cols].notna().sum(axis=1).replace(0, np.nan)

for feat, pattern in position_targets.items():
    idx = pd.Series([
        next((i for i, v in enumerate(row) if pd.notna(v) and pattern in str(v).lower()), -1)
        for row in df_y[ing_cols].values
    ], index=df_y.index)

    df_y[f'{feat}_position'] = np.where(idx >= 0, 1 - idx / total_ing, -1)

# phenoxyethanol 인덱스 (1% 농도 마커)
pheno_idx = pd.Series([
    next((i for i, v in enumerate(row) if pd.notna(v) and 'phenoxyethanol' in str(v).lower()), -1)
    for row in df_y[ing_cols].values
], index=df_y.index)

for feat, pattern in position_targets.items():
    idx = pd.Series([
        next((i for i, v in enumerate(row) if pd.notna(v) and pattern in str(v).lower()), -1)
        for row in df_y[ing_cols].values
    ], index=df_y.index)
    df_y[f'{feat}_above_1pct'] = np.where(
        (idx >= 0) & (pheno_idx >= 0), (idx < pheno_idx).astype(int), -1
    )

# 상위 30% 내 GT 성분 수
df_y['active_top30_count'] = pd.Series([
    sum(1 for pattern in gt_map.values()
        for i, v in enumerate(row[:max(1, int(total_ing.iloc[j]*0.3))])
        if pd.notna(v) and pattern in str(v).lower())
    for j, row in enumerate(df_y[ing_cols].values)
], index=df_y.index)

pos_cols    = [f'{f}_position'   for f in position_targets]
above_cols  = [f'{f}_above_1pct' for f in position_targets]
print("Position 피처 생성 완료")
print(f"  position:   {pos_cols}")
print(f"  above_1pct: {above_cols}")
print(f"  active_top30_count 평균: {df_y['active_top30_count'].mean():.2f}")

Position 피처 생성 완료
  position:   ['niacinamide_position', 'retinol_position', 'vitamin_c_position', 'ceramide_position', 'peptide_position']
  above_1pct: ['niacinamide_above_1pct', 'retinol_above_1pct', 'vitamin_c_above_1pct', 'ceramide_above_1pct', 'peptide_above_1pct']
  active_top30_count 평균: 0.93


In [18]:
# ── 최종 피처 구성 + LightGBM 베이스라인 (position 피처 제외 비교)
mid_feat_cols = [c for c in df_y.columns if c.startswith('mid_')]

feat_cols_no_pos = (
    ['price_low', 'price_mid', 'price_high']
    + ['volume_ml', 'SPF_Index']
    + [c for c in df_y.columns if c.startswith('cat_') and not c.startswith('cat_mid')]
    + mid_feat_cols
    + k_cols + ['k_beauty_ratio']
    + gt_cols + ['us_trend_ratio']
)
feat_cols_no_pos = [c for c in feat_cols_no_pos if c in df_y.columns]

feat_cols_with_pos = (
    feat_cols_no_pos
    + pos_cols + above_cols + ['active_top30_count']
)
feat_cols_with_pos = [c for c in feat_cols_with_pos if c in df_y.columns]

skf  = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
lgbm = LGBMClassifier(random_state=42, verbose=-1)
y    = df_y['target'].values

X_no  = df_y[feat_cols_no_pos].fillna(0).astype(float).values
X_pos = df_y[feat_cols_with_pos].fillna(0).astype(float).values

auc_no  = cross_validate(lgbm, X_no,  y, cv=skf, scoring=['roc_auc'])['test_roc_auc'].mean()
auc_pos = cross_validate(lgbm, X_pos, y, cv=skf, scoring=['roc_auc'])['test_roc_auc'].mean()

print(f"피처 수 (position 제외): {X_no.shape[1]}개   AUC: {auc_no:.4f}  (ml_v4 대비 {auc_no-0.7357:+.4f})")
print(f"피처 수 (position 포함): {X_pos.shape[1]}개   AUC: {auc_pos:.4f}  (ml_v4 대비 {auc_pos-0.7357:+.4f})")
print(f"\nposition 피처 효과: {auc_pos-auc_no:+.4f}")

# 이후 사용할 feat_cols 결정
feat_cols = feat_cols_no_pos if auc_no >= auc_pos else feat_cols_with_pos
X = df_y[feat_cols].fillna(0).astype(float).values
print(f"\n→ 채택: {'position 제외' if auc_no >= auc_pos else 'position 포함'}  ({len(feat_cols)}개 피처)")

피처 수 (position 제외): 48개   AUC: 0.6605  (ml_v4 대비 -0.0752)
피처 수 (position 포함): 59개   AUC: 0.6771  (ml_v4 대비 -0.0586)

position 피처 효과: +0.0166

→ 채택: position 포함  (59개 피처)


In [20]:
# ── GT 커버리지 확인 + 저커버리지 제거 + Importance Pruning
# 1) 커버리지 5% 미만 GT 성분 제거
low_cov = [c for c in gt_cols if df_y[c].mean() < 0.05]
print(f"커버리지 5% 미만 GT 성분 ({len(low_cov)}개 제거): {low_cov}")

feat_cols_pruned = [c for c in feat_cols if c not in low_cov]

# 2) Importance=0 제거
lgbm_full = LGBMClassifier(random_state=42, verbose=-1)
lgbm_full.fit(df_y[feat_cols_pruned].fillna(0).astype(float), y)
imp = pd.Series(lgbm_full.feature_importances_, index=feat_cols_pruned)
zero_imp = imp[imp == 0].index.tolist()
feat_cols_pruned = [c for c in feat_cols_pruned if c not in zero_imp]
print(f"Importance=0 제거 ({len(zero_imp)}개): {zero_imp}")
print(f"최종 피처 수: {len(feat_cols_pruned)}개")

# 3) AUC 비교
X_pruned = df_y[feat_cols_pruned].fillna(0).astype(float).values
auc_pruned = cross_validate(LGBMClassifier(random_state=42, verbose=-1),
                            X_pruned, y, cv=skf, scoring=['roc_auc'])['test_roc_auc'].mean()
print(f"\n정제 후 AUC: {auc_pruned:.4f}  (ml_v4 대비 {auc_pruned-0.7357:+.4f})")

# Top 20 importance (필터 전 imp 재사용)
print("\nTop 20 피처 중요도:")
print(imp[feat_cols_pruned].sort_values(ascending=False).head(20).to_string())

# 이후 사용
feat_cols = feat_cols_pruned
X = X_pruned

커버리지 5% 미만 GT 성분 (9개 제거): ['gt_pdrn', 'gt_exosome', 'gt_nad', 'gt_bakuchiol', 'gt_azelaic_acid', 'gt_tranexamic_acid', 'gt_ectoin', 'gt_centella_asiatica', 'gt_retinol']
Importance=0 제거 (6개): ['mid_sun cream', 'k_galactomyces', 'k_bifida', 'k_snail', 'k_pdrn', 'k_propolis']
최종 피처 수: 44개

정제 후 AUC: 0.6783  (ml_v4 대비 -0.0574)

Top 20 피처 중요도:
volume_ml                    626
us_trend_ratio               327
niacinamide_position         177
active_top30_count           162
peptide_position             150
gt_sodium_hyaluronate        123
gt_tocopherol                112
vitamin_c_position           108
SPF_Index                     94
ceramide_position             90
gt_caprylic                   81
price_mid                     75
gt_tocopheryl_acetate         75
gt_squalane                   66
mid_cream                     66
mid_essence/serum/ampoule     64
price_low                     59
price_high                    48
k_beauty_ratio                47
gt_caffeine                   4

In [21]:
# ── RandomizedSearch 하이퍼파라미터 튜닝
from scipy.stats import randint, uniform

param_dist = {
    'n_estimators':      randint(100, 400),
    'max_depth':         randint(4, 12),
    'num_leaves':        randint(20, 80),
    'learning_rate':     uniform(0.01, 0.09),
    'min_child_samples': randint(10, 40),
    'subsample':         uniform(0.6, 0.4),
    'colsample_bytree':  uniform(0.6, 0.4),
    'reg_alpha':         uniform(0, 0.1),
    'reg_lambda':        uniform(0, 0.1),
}

rs = RandomizedSearchCV(
    LGBMClassifier(random_state=42, verbose=-1),
    param_distributions=param_dist,
    n_iter=100,
    scoring='roc_auc',
    cv=skf,
    random_state=42,
    n_jobs=-1,
)
rs.fit(X, y)
print(f"최적 AUC: {rs.best_score_:.4f}  (ml_v4 대비 {rs.best_score_-0.7357:+.4f})")
print(f"최적 파라미터: {rs.best_params_}")

최적 AUC: 0.7011  (ml_v4 대비 -0.0346)
최적 파라미터: {'colsample_bytree': 0.6727299868828402, 'learning_rate': 0.026506405886809047, 'max_depth': 7, 'min_child_samples': 35, 'n_estimators': 121, 'num_leaves': 63, 'reg_alpha': 0.0023062425041415757, 'reg_lambda': 0.052477466025838915, 'subsample': 0.7599443886861021}
